# Reto 04 — OPT · Análisis de resultados, frentes finales y métricas
### Despacho de la microred (solar + eólico + red)

Este notebook **no ejecuta la búsqueda**: lee los resultados de Optuna guardados por el Notebook 1 y:
1. Calcula el **HV normalizado** (dividiendo por el área del espacio objetivo).
2. Identifica la **mejor configuración** por algoritmo (máx. HV norm, desempate por tiempo).
3. Ejecuta el **frente de Pareto final** de cada algoritmo y compara con métricas (HV, GD, IGD, ε-Indicator, Spread).

> La celda de **configuración y carga de datos** debe coincidir con la del Notebook 1 (misma ventana `T_HOURS`, `START_HOUR`, `PRED_YEAR`).

## 0. Imports y montaje de Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('No estamos en Colab o Drive ya montado:', e)

In [ ]:
!pip install jmetalpy optuna pandas numpy matplotlib

In [ ]:
import os, random, time, json, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from jmetal.core.problem import FloatProblem
from jmetal.core.solution import FloatSolution
from jmetal.algorithm.multiobjective.nsgaii import NSGAII
from jmetal.algorithm.multiobjective.spea2 import SPEA2
from jmetal.operator.crossover import SBXCrossover
from jmetal.operator.mutation import PolynomialMutation
from jmetal.util.termination_criterion import StoppingByEvaluations
from jmetal.util.solution import get_non_dominated_solutions
from jmetal.core.quality_indicator import HyperVolume

warnings.filterwarnings('ignore')
logging.getLogger('jmetal').setLevel(logging.ERROR)
random.seed(42); np.random.seed(42)

## 1. Configuración (debe coincidir con el Notebook 1)

In [ ]:
# ============================================================
#  CONFIGURACIÓN GLOBAL
# ============================================================
BASE_SEED = 42  # Semilla base para reproducibilidad

# --- Rutas (AJUSTA a tu entorno) ------------------------------------------
DATA_DIR   = '/content/drive/MyDrive/reto4-opt/datos'   # carpeta con los 6 CSV
OUTPUT_DIR = '/content/drive/MyDrive/reto4-opt'          # donde se guardan resultados

# --- Horizonte de optimización --------------------------------------------
# El individuo = calendario completo de la ventana.
# Hay 2*T variables: para cada hora t, x1_t (solar) y x2_t (eólico).
# T_HOURS controla el tamaño de la ventana:
#   24 = un día | 168 = una semana | 720 = un mes | 8760 = año completo
T_HOURS    = 168    # ventana representativa (1 semana) -> 336 variables
START_HOUR = 0      # hora de inicio dentro del año (0 .. 8759 - T_HOURS)
PRED_YEAR  = 2020   # año de las predicciones de potencia (2017-2020 disponibles)

# --- Presupuesto evolutivo ------------------------------------------------
# max_evaluations = POPULATION_SIZE * GENERATIONS (igual para NSGA-II y SPEA2)
GENERATIONS     = 250
POPULATION_SIZE = 100   # FIJO: NO se busca en Optuna (decisión del enunciado)
N_RUNS          = 3     # repeticiones por configuración -> se reporta la MEDIANA del HV

# --- Optuna ----------------------------------------------------------------
N_TRIALS   = 40                    # nº de configuraciones que explora Optuna por algoritmo
ALGORITHMS = ['NSGAII', 'SPEA2']

# Espacio de búsqueda CONTINUO (Optuna/TPE afina mejor que un grid discreto)
SEARCH_SPACE = {
    'eta_c'          : (5.0, 30.0),   # índice de distribución SBX  (5=explorador, 30=explotador)
    'eta_m'          : (5.0, 30.0),   # índice de distribución PM
    'crossover_prob' : (0.7, 1.0),    # probabilidad de cruce SBX (literatura: >=0.7)
}

RUTA_BASE = OUTPUT_DIR
print('Configuración cargada.')
print(f'  Ventana            : {T_HOURS} h  (desde la hora {START_HOUR})  -> {2*T_HOURS} variables')
print(f'  Año predicciones   : {PRED_YEAR}')
print(f'  Población (fija)   : {POPULATION_SIZE}')
print(f'  Generaciones       : {GENERATIONS}')
print(f'  Runs por config    : {N_RUNS}')
print(f'  Trials Optuna/algo : {N_TRIALS}')

## 2. Carga y alineación de datos

In [ ]:
# ============================================================
#  CARGA Y ALINEACIÓN DE DATOS
# ============================================================
# IMPORTANTE: los ficheros NO comparten fechas absolutas
#   - precios (solar/eólico/red): año 2025, horario  (8760 h)
#   - predicciones de potencia  : 2017-2020, horario  (~4 años)
#   - demanda (restaurante)     : sin fecha, 8760 valores horarios
# Por eso la alineación es POSICIONAL (hora-del-año, índice 0..8759).
# Tomamos un año de predicciones (PRED_YEAR) y emparejamos por posición.

# --- Demanda d_t (kW = kWh/h) ---------------------------------------------
dem = pd.read_csv(os.path.join(DATA_DIR, 'RefBldgFullServiceRestaurantNew2004_v1_3_7_1_6A_USA_MN_MINNEAPOLIS.csv'))
d_full = dem.iloc[:, 0].to_numpy(dtype=float)            # [kWh/h]

# --- Precio de la red a3_t (€/MWh -> €/kWh) --------------------------------
pg = pd.read_csv(os.path.join(DATA_DIR, 'precio2025-peninsula.csv'), sep=';')
a_grid_full = pg['value'].to_numpy(dtype=float) / 1000.0  # [€/kWh]

# --- Precio solar a1_t y eólico a2_t (€/MWh -> €/kWh) ----------------------
ps = pd.read_csv(os.path.join(DATA_DIR, 'precio_solar_mwh.csv'),  sep=';')
pe = pd.read_csv(os.path.join(DATA_DIR, 'precio_eolico_mwh.csv'), sep=';')
a_solar_full = ps['precio_eur_mwh'].to_numpy(dtype=float) / 1000.0  # [€/kWh]
a_wind_full  = pe['precio_eur_mwh'].to_numpy(dtype=float) / 1000.0  # [€/kWh]

# --- Capacidad solar P_solar_t y eólica P_wind_t (kW = kWh/h) --------------
sol = pd.read_csv(os.path.join(DATA_DIR, 'Predicciones_Solar.csv'))
win = pd.read_csv(os.path.join(DATA_DIR, 'Predicciones_Eolico.csv'))
sol['Date'] = pd.to_datetime(sol['Date'])
win['Date'] = pd.to_datetime(win['Date'])
Psolar_full = sol.loc[sol['Date'].dt.year == PRED_YEAR, 'SystemProduction_AS'].to_numpy(dtype=float)[:8760]
Pwind_full  = win.loc[win['Date'].dt.year == PRED_YEAR, 'Power_AE'].to_numpy(dtype=float)[:8760]

# --- Recorte a longitud común y a la ventana de optimización ---------------
N = min(len(d_full), len(a_grid_full), len(a_solar_full), len(a_wind_full),
        len(Psolar_full), len(Pwind_full))
sl = slice(START_HOUR, START_HOUR + T_HOURS)
assert START_HOUR + T_HOURS <= N, f'La ventana excede los datos disponibles (N={N}).'

DEMAND  = d_full[:N][sl]        # d_t      [kWh/h]
P_SOLAR = Psolar_full[:N][sl]   # P_solar_t [kWh/h]
P_WIND  = Pwind_full[:N][sl]    # P_wind_t  [kWh/h]
A_SOLAR = a_solar_full[:N][sl]  # a1_t [€/kWh]
A_WIND  = a_wind_full[:N][sl]   # a2_t [€/kWh]
A_GRID  = a_grid_full[:N][sl]   # a3_t [€/kWh]

T = len(DEMAND)
print(f'Datos alineados (posicionalmente). Horas en la ventana: T = {T}')
print(f'  Demanda      [kWh/h]: media {DEMAND.mean():.1f}  | max {DEMAND.max():.1f}')
print(f'  Cap. solar   [kWh/h]: media {P_SOLAR.mean():.1f} | max {P_SOLAR.max():.1f}')
print(f'  Cap. eólica  [kWh/h]: media {P_WIND.mean():.1f}  | max {P_WIND.max():.1f}')
print(f'  Precio solar [€/kWh]: media {A_SOLAR.mean():.4f}')
print(f'  Precio eólico[€/kWh]: media {A_WIND.mean():.4f}')
print(f'  Precio red   [€/kWh]: media {A_GRID.mean():.4f}')
print(f'  Horas solar mas barato que red : {(A_SOLAR < A_GRID).mean()*100:.1f}%')
print(f'  Horas eólico mas barato que red: {(A_WIND  < A_GRID).mean()*100:.1f}%')
print(f'  Horas en que renovables cubren la demanda: {((P_SOLAR+P_WIND) >= DEMAND).mean()*100:.1f}%')

## 3. Definición del problema (idéntica al Notebook 1)

In [ ]:
# ============================================================
#  DEFINICIÓN DEL PROBLEMA  (despacho de microred, bi-objetivo)
# ============================================================
# Traducimos las ecuaciones del despacho al lenguaje de jMetalPy.
#
# Variables (2*T):  [x1_0..x1_{T-1} | x2_0..x2_{T-1}]
#   x1_t = energía comprada al solar  en la hora t   (kWh)
#   x2_t = energía comprada al eólico en la hora t   (kWh)
#   g_t  = d_t - x1_t - x2_t  (resto cubierto por la red, >= 0)
#
# Objetivos (ambos MINIMIZAR):
#   f1 = sum_t [ a1_t*x1_t + a2_t*x2_t + a3_t*(d_t - x1_t - x2_t) ]   (coste, €)
#   f2 = sum_t ( d_t - x1_t - x2_t )                                  (energía de red, kWh)
#
# Restricciones:
#   0 <= x1_t <= P_solar_t  ;  0 <= x2_t <= P_wind_t  ;  x1_t + x2_t <= d_t
#
# Estas restricciones se gestionan por REPARACIÓN dentro de evaluate():
#   1) clip de cada flujo a [0, capacidad]  (cotas y no-negatividad)
#   2) si x1_t + x2_t > d_t  ->  reescalado proporcional para que la suma = d_t
# Motivo: dejar la restricción a los objetivos NO funciona. Si x1+x2 > d_t,
# entonces g_t < 0 y BAJARÍA tanto f1 como f2: el infactible "parece" mejor y
# dominaría al factible. La reparación lo evita conservando el reparto aprendido.

class MicrogridDispatch(FloatProblem):
    """Despacho bi-objetivo de una microred (coste vs energía de red)."""

    def __init__(self, demand, p_solar, p_wind, a_solar, a_wind, a_grid):
        super().__init__()
        self.d  = np.asarray(demand,  dtype=float)
        self.ps = np.asarray(p_solar, dtype=float)
        self.pw = np.asarray(p_wind,  dtype=float)
        self.a1 = np.asarray(a_solar, dtype=float)
        self.a2 = np.asarray(a_wind,  dtype=float)
        self.a3 = np.asarray(a_grid,  dtype=float)
        self.T  = len(self.d)

        # Cotas por gen: solar en [0, P_solar_t], eólico en [0, P_wind_t]
        self.lower_bound    = [0.0] * (2 * self.T)
        self.upper_bound    = list(self.ps) + list(self.pw)
        self.obj_directions = [self.MINIMIZE, self.MINIMIZE]
        self.obj_labels     = ['Coste (€)', 'Energia red (kWh)']

    def number_of_variables(self)   -> int: return 2 * self.T
    def number_of_objectives(self)  -> int: return 2
    def number_of_constraints(self) -> int: return 0
    def name(self)                  -> str: return 'Microgrid Dispatch Optimization'

    # ----------------------------------------------------------------
    def _repair(self, variables):
        """Devuelve (x1, x2) factibles a partir del cromosoma."""
        x  = np.asarray(variables, dtype=float)
        x1 = np.clip(x[:self.T],      0.0, self.ps)   # 0 <= x1 <= P_solar
        x2 = np.clip(x[self.T:],      0.0, self.pw)   # 0 <= x2 <= P_wind
        total = x1 + x2
        over  = total > self.d                         # viola x1+x2 <= d_t
        # Reescalado proporcional solo donde se incumple (evita division por 0)
        scale = np.where(over & (total > 0), self.d / np.where(total > 0, total, 1.0), 1.0)
        x1 = x1 * scale
        x2 = x2 * scale
        return x1, x2

    # ----------------------------------------------------------------
    def evaluate(self, solution: FloatSolution) -> FloatSolution:
        x1, x2 = self._repair(solution.variables)
        # Actualizar el cromosoma para que la descendencia herede genes válidos
        solution.variables = np.concatenate([x1, x2]).tolist()

        grid = self.d - x1 - x2                        # energía de red (>= 0)

        f1 = float(self.a1 @ x1 + self.a2 @ x2 + self.a3 @ grid)  # coste total (€)
        f2 = float(grid.sum())                                    # energía de red (kWh)

        solution.objectives[0] = f1
        solution.objectives[1] = f2
        return solution


# Instanciar el problema sobre la ventana cargada
problem = MicrogridDispatch(DEMAND, P_SOLAR, P_WIND, A_SOLAR, A_WIND, A_GRID)
print(f'Problema listo: {problem.name()}')
print(f'Variables: {problem.number_of_variables()}  |  Objetivos: {problem.number_of_objectives()}')

## 4. Cotas del espacio objetivo y punto de referencia del HV

In [ ]:
# ============================================================
#  COTAS DEL ESPACIO OBJETIVO Y PUNTO DE REFERENCIA DEL HV
# ============================================================
# El problema es SEPARABLE hora a hora, así que los extremos de cada objetivo
# se calculan analíticamente (exactos, deterministas):
#
#   f2 (energía de red):
#     min = sum_t max(0, d_t - P_solar_t - P_wind_t)   (renovables al máximo)
#     max = sum_t d_t                                  (todo de la red)
#   f1 (coste): por hora se rellena la demanda
#     min -> fuentes mas baratas que la red primero
#     max -> fuentes mas caras que la red primero  (peor rincón factible)
#
# El punto de referencia del HV es el peor rincón (f1_max, f2_max) + margen,
# que GARANTIZA dominar todo el frente (mejor que muestrear: el muestreo aleatorio
# tiende a sobreusar renovables e infraestima el extremo de mucha energía de red).

def objective_bounds(d, ps, pw, a_s, a_w, a_g):
    f1_min = f1_max = 0.0
    for t in range(len(d)):
        srcs = [(a_s[t], ps[t]), (a_w[t], pw[t])]
        rem, c = d[t], 0.0                                   # coste mínimo
        for price, cap in sorted(srcs, key=lambda z: z[0]):
            if price < a_g[t]:
                u = min(cap, rem); c += price * u; rem -= u
        c += a_g[t] * rem; f1_min += c
        rem, c = d[t], 0.0                                   # coste máximo
        for price, cap in sorted(srcs, key=lambda z: -z[0]):
            if price > a_g[t]:
                u = min(cap, rem); c += price * u; rem -= u
        c += a_g[t] * rem; f1_max += c
    f2_min = float(np.maximum(0.0, d - ps - pw).sum())
    f2_max = float(d.sum())
    return f1_min, f1_max, f2_min, f2_max

MARGIN = 0.1
F1_MIN, F1_MAX, F2_MIN, F2_MAX = objective_bounds(DEMAND, P_SOLAR, P_WIND, A_SOLAR, A_WIND, A_GRID)
area_espacio    = (F1_MAX - F1_MIN) * (F2_MAX - F2_MIN)
REFERENCE_POINT = np.array([F1_MAX, F2_MAX]) * (1 + MARGIN)

print(f'f1 (coste)       : min {F1_MIN:.4f} €   | max {F1_MAX:.4f} €')
print(f'f2 (energia red) : min {F2_MIN:.2f} kWh | max {F2_MAX:.2f} kWh')
print(f'Área espacio objetivo: {area_espacio:.4f}')
print(f'Punto de referencia HV: coste={REFERENCE_POINT[0]:.4f} €, energia_red={REFERENCE_POINT[1]:.2f} kWh')

## 5. Carga de resultados de Optuna

In [ ]:
# ============================================================
#  CARGA DE RESULTADOS DEL NOTEBOOK 1
# ============================================================
df = pd.read_csv(os.path.join(RUTA_BASE, 'resultados_optuna_microred.csv'))
print(f'Filas (trials): {len(df)}')
print(f'Columnas      : {list(df.columns)}')
print(f'Algoritmos    : {df["algorithm"].unique()}')
df.head(3)

## 6. HV normalizado

In [ ]:
# ============================================================
#  HV NORMALIZADO
# ============================================================
# HV_norm = HV_raw / area_espacio  -> valor comparable entre configuraciones.
df['hv_norm']     = df['hv_median'] / area_espacio
df['hv_norm_std'] = df['hv_std']    / area_espacio
print('Rango HV raw :', df['hv_median'].min(), '->', df['hv_median'].max())
print('Rango HV norm:', df['hv_norm'].min(),   '->', df['hv_norm'].max())

## 7. Top configuraciones y mejor por algoritmo

In [ ]:
# ============================================================
#  TOP CONFIGURACIONES POR ALGORITMO
# ============================================================
cols = ['algorithm', 'population_size', 'eta_c', 'eta_m',
        'crossover_prob', 'hv_norm', 'hv_norm_std', 'elapsed_mean_s']
df_sorted = df.sort_values(['algorithm', 'hv_norm', 'elapsed_mean_s'],
                           ascending=[True, False, True])
print('=== Top 10 configuraciones por algoritmo (HV normalizado) ===')
display(df_sorted[cols].groupby('algorithm').head(10))

In [ ]:
# ============================================================
#  MEJOR CONFIGURACIÓN POR ALGORITMO  (máx HV norm, desempate por tiempo)
# ============================================================
best_configs = {}
for algo in df['algorithm'].unique():
    da = df[df['algorithm'] == algo]
    mejor = da['hv_norm'].max()
    empates = da[da['hv_norm'] == mejor]
    fila = empates.sort_values('elapsed_mean_s').iloc[0]   # menor tiempo
    best_configs[algo] = fila.to_dict()
    print(f'{algo}: eta_c={fila.eta_c:.3f}  eta_m={fila.eta_m:.3f}  '
          f'p_c={fila.crossover_prob:.3f}  pop={int(fila.population_size)}  '
          f'HV_norm={fila.hv_norm:.6f}')

## 8. Visualizaciones

In [ ]:
# ============================================================
#  SENSIBILIDAD: HV normalizado vs cada hiperparámetro (continuo)
# ============================================================
hp_params = ['eta_c', 'eta_m', 'crossover_prob']
hp_labels = ['η_c (SBX)', 'η_m (PM)', 'Prob. cruce']
algorithms = list(df['algorithm'].unique())
colors = {'NSGAII': '#1f77b4', 'SPEA2': '#ff7f0e'}

fig, axes = plt.subplots(len(algorithms), len(hp_params),
                         figsize=(15, 4.5 * len(algorithms)), sharey='row')
if len(algorithms) == 1:
    axes = axes.reshape(1, -1)

for row, algo in enumerate(algorithms):
    da = df[df['algorithm'] == algo]
    for col, (p, lbl) in enumerate(zip(hp_params, hp_labels)):
        ax = axes[row, col]
        ax.scatter(da[p], da['hv_norm'], color=colors.get(algo, 'gray'), alpha=0.7, s=35)
        ax.set_xlabel(lbl); ax.set_title(f'{algo}')
        if col == 0:
            ax.set_ylabel('HV normalizado')
        ax.grid(alpha=0.3)
plt.suptitle('Influencia de cada hiperparámetro sobre el HV normalizado', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  COMPARATIVA NSGA-II vs SPEA2: HV normalizado vs tiempo
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
for algo in algorithms:
    da = df[df['algorithm'] == algo].sort_values('elapsed_mean_s')
    ax.scatter(da['elapsed_mean_s'], da['hv_norm'], label=algo,
               color=colors.get(algo, 'gray'), alpha=0.7, s=35)
ax.set_xlabel('Tiempo medio por run (s)'); ax.set_ylabel('HV normalizado')
ax.set_title('Calidad del frente vs coste computacional'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Ejecución final y frentes de Pareto

In [ ]:
# ============================================================
#  EJECUCIÓN FINAL CON LA MEJOR CONFIGURACIÓN DE CADA ALGORITMO
# ============================================================
MAX_EVALUATIONS = POPULATION_SIZE * GENERATIONS
pareto_fronts = {}

for algo_name, cfg in best_configs.items():
    pop_size = int(cfg['population_size'])
    eta_c    = float(cfg['eta_c'])
    eta_m    = float(cfg['eta_m'])
    cx_prob  = float(cfg['crossover_prob'])
    mut_prob = 1.0 / problem.number_of_variables()

    crossover = SBXCrossover(probability=cx_prob, distribution_index=eta_c)
    mutation  = PolynomialMutation(probability=mut_prob, distribution_index=eta_m)
    criterion = StoppingByEvaluations(max_evaluations=MAX_EVALUATIONS)

    if algo_name == 'NSGAII':
        algorithm = NSGAII(problem=problem, population_size=pop_size,
                           offspring_population_size=pop_size,
                           mutation=mutation, crossover=crossover,
                           termination_criterion=criterion)
    elif algo_name == 'SPEA2':
        algorithm = SPEA2(problem=problem, population_size=pop_size,
                          offspring_population_size=pop_size,
                          mutation=mutation, crossover=crossover,
                          termination_criterion=criterion)
    else:
        continue

    print(f'Ejecutando {algo_name} (pop={pop_size}, η_c={eta_c:.2f}, η_m={eta_m:.2f}, p_c={cx_prob:.2f})...')
    t0 = time.time(); algorithm.run(); elapsed = time.time() - t0
    front = get_non_dominated_solutions(algorithm.result())
    pareto_fronts[algo_name] = front
    print(f'  Terminado en {elapsed:.1f}s | soluciones en el frente: {len(front)}')

print('\nFrentes guardados para:', list(pareto_fronts.keys()))

In [ ]:
# ============================================================
#  FRENTES DE PARETO SUPERPUESTOS
# ============================================================
fig, ax = plt.subplots(figsize=(9, 6))
colors_front = {'NSGAII': '#1f77b4', 'SPEA2': '#ff7f0e'}
markers      = {'NSGAII': 'o',       'SPEA2': 's'}

for algo_name, front in pareto_fronts.items():
    f1 = [s.objectives[0] for s in front]   # coste
    f2 = [s.objectives[1] for s in front]   # energía de red
    order = sorted(range(len(f1)), key=lambda i: f1[i])
    ax.plot([f1[i] for i in order], [f2[i] for i in order],
            marker=markers.get(algo_name, 'o'), linestyle='-',
            color=colors_front.get(algo_name, 'gray'), label=algo_name, alpha=0.8)

ax.set_xlabel('f1 — Coste total (€)')
ax.set_ylabel('f2 — Energía de red (kWh)')
ax.set_title(f'Frentes de Pareto (ventana de {T} h)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  SOLUCIONES REPRESENTATIVAS DEL FRENTE
# ============================================================
def front_to_dataframe(front) -> pd.DataFrame:
    data = []
    for s in front:
        x  = np.asarray(s.variables, dtype=float)
        x1 = x[:T]; x2 = x[T:]
        data.append({
            'coste_eur'        : s.objectives[0],
            'energia_red_kwh'  : s.objectives[1],
            'energia_solar_kwh': float(x1.sum()),
            'energia_eolico_kwh': float(x2.sum()),
            'pct_renovable'    : 100.0 * float((x1.sum() + x2.sum()) / DEMAND.sum()),
        })
    d = pd.DataFrame(data)
    return d.sort_values('coste_eur').reset_index(drop=True) if not d.empty else d

def representative(df_f, label):
    if df_f.empty:
        return df_f
    idx_min_coste = df_f['coste_eur'].idxmin()
    idx_min_red   = df_f['energia_red_kwh'].idxmin()
    idx_mid       = len(df_f) // 2
    sel = df_f.loc[[idx_min_coste, idx_mid, idx_min_red]].copy()
    sel.index = ['Mínimo coste', 'Equilibrada', 'Mínima energía de red']
    return sel

for algo_name, front in pareto_fronts.items():
    print(f'\n=== {algo_name} ===')
    display(representative(front_to_dataframe(front), algo_name))

## 10. Métricas de calidad

In [ ]:
# ============================================================
#  MÉTRICAS DE CALIDAD POR ALGORITMO
# ============================================================
from jmetal.core.quality_indicator import (
    HyperVolume, GenerationalDistance, InvertedGenerationalDistance, EpsilonIndicator)

def spread(front, reference_front):
    front = np.array(front); reference_front = np.array(reference_front)
    front = front[np.argsort(front[:, 0])]
    reference_front = reference_front[np.argsort(reference_front[:, 0])]
    d_f = np.linalg.norm(front[0]  - reference_front[0])
    d_l = np.linalg.norm(front[-1] - reference_front[-1])
    distances = np.linalg.norm(np.diff(front, axis=0), axis=1)
    if len(distances) == 0:
        return 0.0
    d_mean = np.mean(distances)
    return (d_f + d_l + np.sum(np.abs(distances - d_mean))) / (d_f + d_l + len(distances) * d_mean)

# Frente de referencia = no dominadas de la unión de ambos algoritmos
todas = []
for fr in pareto_fronts.values():
    todas.extend(fr)
ref_sol = get_non_dominated_solutions(todas)
reference_front_array = np.array([s.objectives for s in ref_sol])

print('=' * 60)
print('MÉTRICAS DE CALIDAD — mejor configuración por algoritmo')
print(f'Punto de referencia HV: {REFERENCE_POINT}')
print(f'Frente de referencia global: {len(reference_front_array)} soluciones')
print('=' * 60)

resumen = {}
for algo_name, front in pareto_fronts.items():
    fa  = np.array([s.objectives for s in front])
    hv  = HyperVolume(reference_point=REFERENCE_POINT.tolist())
    gd  = GenerationalDistance(reference_front=reference_front_array)
    igd = InvertedGenerationalDistance(reference_front=reference_front_array)
    eps = EpsilonIndicator(reference_front=reference_front_array)
    resumen[algo_name] = {
        'HV raw'        : hv.compute(fa),
        'HV normalizado': hv.compute(fa) / area_espacio,
        'GD'            : gd.compute(fa),
        'IGD'           : igd.compute(fa),
        'ε-Indicator'   : eps.compute(fa),
        'Spread'        : spread(fa, reference_front_array),
        'Nº soluciones' : len(front),
    }

dfm = pd.DataFrame(resumen).T
dfm.index.name = 'Algoritmo'
display(dfm.style
        .format('{:.6f}', subset=[c for c in dfm.columns if c != 'Nº soluciones'])
        .format('{:.0f}', subset=['Nº soluciones'])
        .background_gradient(subset=['HV normalizado'], cmap='YlGn')
        .background_gradient(subset=['GD', 'IGD', 'ε-Indicator', 'Spread'], cmap='YlOrRd_r')
        .set_caption('Métricas de calidad del frente de Pareto'))